# PSNR

Peak Signal-to-Noise Ratio. Mide la relación entre la señal máxima posible y el ruido introducido por la compresión. Se basa en el MSE entre imagen original y distorsionada. Valores más altos indican mejor calidad (típicamente 30–50 dB para buena calidad).

In [1]:
from skimage import io, color
from skimage.metrics import peak_signal_noise_ratio

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:  # RGBA
        return color.rgba2rgb(img)
    return img

def calcular_psnr(img_orig_path, img_comp_path):
    orig = safe_rgb(io.imread(img_orig_path))
    comp = safe_rgb(io.imread(img_comp_path))
    
    orig = color.rgb2gray(orig)
    comp = color.rgb2gray(comp)
    
    return peak_signal_noise_ratio(orig, comp, data_range=1.0)

In [2]:
print(calcular_psnr('cute.png', 'imagenes/cute.jpg'))


46.3892106695685


# SSIM

Structural Similarity Index. Mide similitud entre dos imágenes comparando luminancia, contraste y estructura. Resultado entre -1 y 1; donde 1 = similitud perfecta. Mejor correlación con percepción humana que PSNR.

In [4]:
from skimage import io
from skimage.metrics import structural_similarity

def calcular_ssim(img_orig_path, img_comp_path):
    orig = safe_rgb(io.imread(img_orig_path))
    comp = safe_rgb(io.imread(img_comp_path))
    
    return structural_similarity(
        orig, comp,
        channel_axis=2,
        data_range=1.0
    )

In [5]:
score= calcular_ssim('cute.png', 'imagenes/cute.jpg')   
print(f'SSIM global: {score}')

SSIM global: 0.04886610868169924


# MS-SSIM

Multi-Scale Structural Similarity. Extensión de SSIM que evalúa similitud estructural en múltiples escalas espaciales (resoluciones). Captura degradaciones visuales tanto en detalles finos como en estructuras globales.

In [6]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io, color
from pytorch_msssim import ms_ssim

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:  # RGBA
        return color.rgba2rgb(img)
    return img

def load_tensor(path):
    img = safe_rgb(io.imread(path))
    img = img.astype(np.float32) / 255.0
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# MS-SSIM
# ---------------------------

def calcular_ms_ssim(img_orig_path, img_comp_path):
    orig = load_tensor(img_orig_path)
    comp = load_tensor(img_comp_path)
    
    orig, comp = match_size(orig, comp)
    
    score = ms_ssim(orig, comp, data_range=1.0, size_average=True)
    return score.item()

In [7]:
print(calcular_ms_ssim('cute.png', 'imagenes/cute.jpg'))

0.18311285972595215


# LPIPS


Learned Perceptual Image Patch Similarity. Mide diferencia perceptual usando características extraídas de redes neuronales (VGG/AlexNet). Valores más bajos = más similares. Alta correlación con percepción humana.

In [8]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io, color
import lpips

# ---------------------------
# CONFIGURACIÓN GLOBAL (evita recargar modelo cada vez)
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn = lpips.LPIPS(net='vgg').to(device)
loss_fn.eval()

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:  # RGBA
        return color.rgba2rgb(img)
    return img

def load_tensor(path):
    img = safe_rgb(io.imread(path))
    img = img.astype(np.float32) / 127.5 - 1.0  # [-1, 1]
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# LPIPS
# ---------------------------

def calcular_lpips(img_orig_path, img_comp_path):
    orig = load_tensor(img_orig_path)
    comp = load_tensor(img_comp_path)
    
    orig, comp = match_size(orig, comp)
    
    with torch.no_grad():
        dist = loss_fn(orig, comp)
    
    return dist.item()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth


In [9]:
print(calcular_lpips('cute.png', 'imagenes/cute.jpg'))

0.6470681428909302


# VIF

Visual Information Fidelity. Mide cuánta información visual del original se conserva en la imagen distorsionada, basándose en teoría de la información y modelos del sistema visual humano.

In [10]:
import numpy as np
from skimage import io, color
from sewar.full_ref import vifp
from skimage.transform import resize

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_gray(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    return img

def load_gray(path):
    img = io.imread(path)
    img = safe_gray(img)
    return img.astype(np.float64)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# VIF (REAL)
# ---------------------------

def calcular_vif(img_orig_path, img_comp_path):
    orig = load_gray(img_orig_path)
    comp = load_gray(img_comp_path)
    
    orig, comp = match_size(orig, comp)
    
    return vifp(orig, comp)

In [11]:
print(calcular_vif('cute.png', 'imagenes/cute.jpg'))

0.9944796332450144


# FSIM

Feature Similarity Index. Evalúa similitud perceptual usando congruencia de fase (Phase Congruency) y magnitud del gradiente. Muy efectivo para detectar degradaciones en bordes, contornos y texturas.

In [41]:
import numpy as np
from skimage import io, color
from scipy.ndimage import sobel
from skimage.transform import resize

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_gray(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    return img

def load_gray(path):
    img = io.imread(path)
    img = safe_gray(img)
    return img.astype(np.float64)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# FSIM APROXIMADO
# ---------------------------

def calcular_fsim_aproximado(img_orig_path, img_comp_path, T1=0.85, T2=160):
    orig = load_gray(img_orig_path)
    comp = load_gray(img_comp_path)
    
    orig, comp = match_size(orig, comp)

    def grad_mag(img):
        gx = sobel(img, axis=1)
        gy = sobel(img, axis=0)
        return np.sqrt(gx**2 + gy**2)

    G1 = grad_mag(orig)
    G2 = grad_mag(comp)

    # Similaridad de gradiente
    GMS = (2 * G1 * G2 + T2) / (G1**2 + G2**2 + T2)

    # Similaridad de luminancia
    SL = (2 * orig * comp + T1) / (orig**2 + comp**2 + T1)

    return np.mean(GMS * SL)

In [42]:
print(calcular_fsim_aproximado('cute.png', 'imagenes/cute.jpg'))

0.9999810116784077


# GMSD

Gradient Magnitude Similarity Deviation. Compara la magnitud de gradientes entre imagen original y distorsionada. Bajo costo computacional y buena correlación perceptual. Valores más bajos = mejor calidad.

In [43]:
import numpy as np
from skimage import io, color
from scipy.ndimage import sobel
from skimage.transform import resize

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

def safe_gray(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    return img

def load_gray(path):
    img = io.imread(path)
    img = safe_gray(img)
    return img.astype(np.float64)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = resize(img2, img1.shape, anti_aliasing=True)
    return img1, img2

# ---------------------------
# GMSD
# ---------------------------

def calcular_gmsd(img_orig_path, img_comp_path, C=0.0026):
    orig = load_gray(img_orig_path)
    comp = load_gray(img_comp_path)
    
    orig, comp = match_size(orig, comp)

    def grad_mag(arr):
        gx = sobel(arr, axis=1)
        gy = sobel(arr, axis=0)
        return np.sqrt(gx**2 + gy**2)

    Gx = grad_mag(orig)
    Gy = grad_mag(comp)

    # Similaridad de gradiente
    GMS = (2 * Gx * Gy + C) / (Gx**2 + Gy**2 + C)

    # Desviación estándar del mapa
    return np.std(GMS)

In [ ]:
print(calcular_gmsd('cute.png', 'imagenes/cute.jpg'))

0.06317453712437866


# MS-SSIM + LPIPS (combinado)

Enfoque híbrido que combina MS-SSIM (similitud estructural multiescala) y LPIPS (diferencia perceptual profunda). Captura simultáneamente distorsiones estructurales globales y diferencias perceptuales de alto nivel.

In [16]:
import torch
import lpips
import numpy as np
from PIL import Image
from pytorch_msssim import ms_ssim

def calcular_ms_ssim_lpips(img_orig_path, img_comp_path,
                            alpha=0.5, beta=0.5):
    loss_fn = lpips.LPIPS(net='vgg')

    arr_o = np.array(Image.open(img_orig_path).convert('RGB'),
                     dtype=np.float32)
    arr_c = np.array(Image.open(img_comp_path).convert('RGB'),
                     dtype=np.float32)

    # Tensor normalizado [0,1] para MS-SSIM
    t_o = torch.from_numpy(arr_o/255.0).permute(2,0,1).unsqueeze(0)
    t_c = torch.from_numpy(arr_c/255.0).permute(2,0,1).unsqueeze(0)
    ms = ms_ssim(t_o, t_c, data_range=1.0,
                 size_average=True).item()

    # Tensor normalizado [-1,1] para LPIPS
    lp_o = torch.from_numpy(arr_o/127.5 - 1).permute(2,0,1).unsqueeze(0)
    lp_c = torch.from_numpy(arr_c/127.5 - 1).permute(2,0,1).unsqueeze(0)
    with torch.no_grad():
        lp = loss_fn(lp_o, lp_c).item()

    # Score combinado (mayor = mejor)
    score = alpha * ms + beta * (1 - lp)
    return {'ms_ssim': ms, 'lpips': lp, 'combinado': score}

# VMAF

Video Multi-Method Assessment Fusion. Desarrollado por Netflix. Combina VIF, detalle y SSIM mediante un modelo SVM entrenado con evaluaciones humanas. Resultado 0–100, donde mayor = mejor calidad. Principalmente para video, también aplicable a imágenes.

In [1]:
import subprocess
import json
import os

def calcular_vmaf_ffmpeg(ref_path, dist_path, log_path="vmaf_log.json"):
    try:
        # Comando FFmpeg con libvmaf
        cmd = [
            'ffmpeg',
            '-i', dist_path,
            '-i', ref_path,
            '-lavfi', f'libvmaf=log_fmt=json:log_path={log_path}',
            '-f', 'null', '-'
        ]

        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        # Verificar errores
        if result.returncode != 0:
            print("[ERROR FFmpeg]")
            print(result.stderr)
            return None

        # Leer resultado
        if not os.path.exists(log_path):
            print("[ERROR] No se generó el log de VMAF")
            return None

        with open(log_path) as f:
            log = json.load(f)

        return log['pooled_metrics']['vmaf']['mean']

    except Exception as e:
        print(f"[ERROR] {e}")
        return None